# Análisis exploratorio de datos (EDA)
## Intentos de suicidio notificados en Tunja, 2020-2024

**Proyecto integrador – Primer semestre**
Asignaturas: Introducción a la Ingeniería de Datos e IA · Lógica Computacional

**Pregunta guía:** ¿Quiénes son las personas con intentos de suicidio notificados en Tunja entre 2020 y 2024,
qué situaciones las rodean y cómo ha cambiado la cantidad de casos?

> ⚠️ Tema sensible. Los datos no contienen nombres ni documentos de identidad y se analizan solo de forma agregada.

## 1. Carga de datos
En Google Colab: sube `Tunja_intentos_datos_limpios.csv` con el panel de archivos (📁) o desde Google Drive.
La celda busca el archivo en las rutas más comunes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidatas = [
    Path("Tunja_intentos_datos_limpios.csv"),            # Colab: archivo subido
    Path("../data/Tunja_intentos_datos_limpios.csv"),    # repositorio local
    Path("data/Tunja_intentos_datos_limpios.csv"),
]
ruta = next(p for p in candidatas if p.exists())
df = pd.read_csv(ruta, parse_dates=["FECHA_NOTIFICACION", "FECHA_HECHO"])
df = df.rename(columns={"ANIO": "AÑO"})
print(f"Archivo: {ruta}  ->  {df.shape[0]} registros y {df.shape[1]} variables")

## 2. Primer contacto con los datos: `head`, `info`, `describe`

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object').T[['count', 'unique', 'top', 'freq']]

## 3. Calidad y limpieza
Revisamos valores vacíos, duplicados y fechas inconsistentes.

In [ ]:
vacios = df.isna().sum()
print("Columnas con vacíos:")
print(vacios[vacios > 0])
print("\nIDs duplicados:", df["ID_REGISTRO"].duplicated().sum())
print("Fechas del hecho posteriores a la notificación:", (df["FECHA_HECHO"] > df["FECHA_NOTIFICACION"]).sum())

In [ ]:
# Meses sin notificaciones por año (bucle + set)
df["MES"] = df["FECHA_NOTIFICACION"].dt.month
for anio, sub in df.groupby("AÑO"):
    faltan = [m for m in range(1, 13) if m not in set(sub["MES"])]
    estado = "completo" if not faltan else f"faltan meses {faltan}"
    print(anio, "->", estado)

**Hallazgo de calidad:** 2023 no tiene notificaciones de julio a septiembre y 2024 solo llega a marzo.
Por eso las comparaciones entre años se hacen solo con años completos (2020-2022).

In [ ]:
# Variable derivada con condicionales: grupo de edad
def clasificar_edad(edad):
    if edad <= 11:
        return "Niñez (6-11)"
    elif edad <= 17:
        return "Adolescencia (12-17)"
    elif edad <= 28:
        return "Juventud (18-28)"
    elif edad <= 59:
        return "Adultez (29-59)"
    else:
        return "Mayores (60+)"

df["GRUPO_EDAD"] = df["EDAD"].apply(clasificar_edad)
df["GRUPO_EDAD"].value_counts()

## 4. Estadísticas descriptivas con NumPy

In [ ]:
edad = df["EDAD"].to_numpy(dtype=float)
intentos = df["NUMERO_INTENTOS"].dropna().to_numpy(dtype=float)

def resumen(x):
    return {
        "media": np.mean(x), "mediana": np.median(x), "desviacion": np.std(x, ddof=1),
        "minimo": np.min(x), "maximo": np.max(x),
        "q1": np.percentile(x, 25), "q3": np.percentile(x, 75),
    }

pd.DataFrame({"Edad": resumen(edad), "Número de intentos": resumen(intentos)}).round(2)

In [ ]:
casos_anio = df.groupby("AÑO").size()
variacion = np.diff(casos_anio.to_numpy()) / casos_anio.to_numpy()[:-1] * 100
print(casos_anio)
print("\nVariación % año a año:", dict(zip(casos_anio.index[1:], variacion.round(1).tolist())))
print("Correlación edad vs. n.º de intentos:",
      np.corrcoef(df.dropna(subset=["NUMERO_INTENTOS"])[["EDAD", "NUMERO_INTENTOS"]].to_numpy().T)[0, 1].round(3))

## 5. Visualizaciones iniciales

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))

casos_anio.plot(kind="bar", ax=ejes[0], color="#2A7F8E")
ejes[0].set_title("Casos notificados por año")
ejes[0].set_xlabel("")

ejes[1].hist(edad, bins=30, color="#3E5C76")
ejes[1].axvline(np.median(edad), color="#D08C2E", linestyle="--", label="Mediana")
ejes[1].set_title("Distribución de la edad")
ejes[1].legend()

df["SEXO"].value_counts().plot(kind="pie", ax=ejes[2], autopct="%1.0f%%",
                               colors=["#2A7F8E", "#3E5C76"])
ejes[2].set_title("Sexo")
ejes[2].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
factores = ["PROBLEMAS_FAMILIARES", "CONFLICTO_CON_PAREJA_O_EX_PAREJA", "ESCOLAR_EDUCATIVA",
            "PROBLEMAS_ECONOMICOS", "MALTRATO_FISICO_SICOLOGICO_SEXUAL", "PROBLEMAS_LABORALES",
            "MUERTE_DE_UN_FAMILIAR", "ENFERMEDAD_CRONICA_DOLOROSA_O_DISCAPACITANTE"]
pct = {f.replace("_", " ").capitalize(): (df[f] == "SI").mean() * 100 for f in factores}
pd.Series(pct).sort_values().plot(kind="barh", figsize=(8, 4), color="#2A7F8E",
                                  title="% de casos con cada factor desencadenante")
plt.xlabel("% de casos")
plt.show()

## 6. Hallazgos preliminares
1. **Quiénes:** cerca de 2 de cada 3 casos son mujeres; la edad mediana es 21 años y 3 de cada 4 casos
   tienen entre 12 y 28 años.
2. **Qué los rodea:** los factores más registrados son los problemas familiares y el conflicto con la pareja;
   más de la mitad tiene antecedente de trastorno psiquiátrico y la mitad ya había tenido un intento previo.
3. **Cuándo:** entre 2020 y 2022 (años completos) los casos notificados aumentaron cerca de 79%.
4. **Calidad:** 17% de vacíos en `NUMERO_INTENTOS`, 22 registros con fechas inconsistentes y meses sin
   datos en 2023-2024.

## 7. Reflexión: datos e IA
Los datos de vigilancia permiten ver un problema que suele ser silencioso, pero solo muestran los casos que
llegan al sistema de salud (sub-registro). Un modelo de IA podría ayudar a priorizar seguimiento a personas
con varias señales de riesgo, pero también podría estigmatizar o reproducir sesgos; su uso debe ser
anónimo, transparente y siempre con decisión humana.

Estos hallazgos se desarrollan de forma interactiva en el dashboard de Streamlit (`app.py`).